In [ ]:
import pandas as pd
dataset = 'rotten_tomatoes'
import matplotlib.pyplot as plt
import numpy as np
import os
import json

In [ ]:
def plot_prevalence(dist, info, title, out):
    k = dist.shape[1]
    if len(info) != k:
        raise ValueError(f"{len(info)} topics in topic_info but {k} columns in the distribution; "
                         "are both files from the same fold?")
    soft = dist.mean(axis=0)
    hard = info.set_index("Topic")["Count"].reindex(range(k)).to_numpy(float)
    hard /= hard.sum()

    order = np.argsort(soft)              # largest topic on top
    y = np.arange(k)
    h = 0.4
    fig, ax = plt.subplots(figsize=(6.5, 0.35 * k + 1.4))
    ax.barh(y + h / 2, soft[order], h, label=r"Soft share (mean $p(t\mid r)$)", color="#3b6ea5")
    ax.barh(y - h / 2, hard[order], h, label="Hard share (K-Means cluster size)", color="#c8a45a")
    ax.set_yticks(y)
    # ax.set_yticklabels([labels[t] for t in order])
    ax.set_xlabel("Share of reviews")
    if title:
        ax.set_title(title)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(f"{out}_prevalence.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig)


In [ ]:
def plot_keywords(model_dir, n_keywords, out, ncols=3):
    path = os.path.join(model_dir, "topics.json")
    with open(path) as f:
        reps = json.load(f)["topic_representations"]
    topics = sorted(int(t) for t in reps if int(t) != -1)

    nrows = int(np.ceil(len(topics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 1.9 * nrows), squeeze=False)
    for ax, t in zip(axes.flat, topics):
        words = [(w, s) for w, s in reps[str(t)] if w][:n_keywords][::-1]
        ax.barh([w for w, _ in words], [s for _, s in words], color="#3b6ea5")
        # ax.set_title(labels.get(t, f"T{t}"), fontsize=8)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes.flat[len(topics):]:
        ax.axis("off")
    fig.supxlabel("Keyword weight (cosine similarity to topic embedding)", fontsize=8)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.show()
        fig.savefig(f"{out}_keywords.{ext}", dpi=300, bbox_inches="tight")

    plt.close(fig)

In [ ]:
df_topics_info = pd.read_csv(f'{dataset}/fold2/topic_info.csv')
df_topics_dist = np.load(f'{dataset}/fold2/review_topic_dist.npy')

plot_prevalence(df_topics_dist, df_topics_info, None, '')
plot_keywords(f'{dataset}/fold2/bertopic_model', 10, '')

In [4]:
df_topics_fold2 = pd.read_csv(f'{dataset}/fold2/topic_info.csv')
df_topics_search_fold2 = pd.read_csv(f'{dataset}/fold2/topic_search.csv')

In [5]:
df_topics_fold3 = pd.read_csv(f'{dataset}/fold3/topic_info.csv')
df_topics_search_fold3 = pd.read_csv(f'{dataset}/fold3/topic_search.csv')

In [6]:
df_topics_fold4 = pd.read_csv(f'{dataset}/fold4/topic_info.csv')
df_topics_search_fold4 = pd.read_csv(f'{dataset}/fold4/topic_search.csv')

In [14]:
df_topics_search_fold4

,embedding_model,k,c_v,c_npmi,topic_diversity,min_topic_size,fit_seconds
0,sentence-transformers/all-mpnet-base-v2,15,0.356005,-0.206373,0.890667,390,59.4
1,sentence-transformers/all-mpnet-base-v2,20,0.351181,-0.240697,0.908000,390,59.6
2,sentence-transformers/all-mpnet-base-v2,25,0.348440,-0.248908,0.913600,390,62.1
3,sentence-transformers/all-mpnet-base-v2,10,0.339374,-0.181542,0.852000,9193,56.3
4,sentence-transformers/all-mpnet-base-v2,30,0.337009,-0.222243,0.912517,360,62.3
